# 02 — Train: U-Net parcel boundary segmentation (Colab)

**Runtime: GPU (T4 minimum; L4 / A100 faster).** Runtime menu → Change runtime type → GPU.

Prereq: `01_prep_colab.ipynb` has run and `/content/drive/MyDrive/aigeolab_train/{tiles, labels, manifest.csv}` exists.

Pipeline (v1.1 — eliminates per-batch TIFF I/O):
1. Install deps + mount Drive + clone repo.
2. Load config; resolve Colab paths.
3. Read manifest; mouza-disjoint train/val split.
4. Rasterise polygons → 3-px boundary mask (cached to `staging/masks/`).
5. Patch each 10000×10000 tile into 512×512 windows; drop empty patches.
6. Copy tiles + masks to Colab's local SSD.
7. **Pre-extract all patches into RAM** (read each TIFF once, slice all patches from memory). Eliminates the per-batch TIFF-open bottleneck that capped throughput at ~10 min/epoch.
8. In-memory Dataset + DataLoaders.
9. Train U-Net (ResNet-34, BCE+Dice, AdamW+cosine, AMP).
10. Qualitative eval overlay on a held-out tile.

In [ ]:
# --- Cell 1: install deps + mount Drive + clone repo ---
!pip install -q rasterio shapely segmentation-models-pytorch albumentations opencv-python-headless pyyaml pyshp

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO_URL = 'https://github.com/tahmid013/AIGEOLAB_OFFICE.git'
REPO_DIR = '/content/AIGEOLAB_OFFICE'
if os.path.isdir(REPO_DIR):
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True).stdout)
os.chdir(REPO_DIR)


In [ ]:
# --- Cell 2: load config, resolve Colab paths ---
import yaml
from pathlib import Path

with open('config.yaml') as f: CFG = yaml.safe_load(f)
ENV = 'colab'
P = CFG['paths'][ENV]
STAGE = Path(P['staging_root'])
assert STAGE.is_dir(), f'{STAGE} not found. Run 01_prep_colab.ipynb first.'
TILES   = STAGE / 'tiles'
LABELS  = STAGE / 'labels'
MASKS   = STAGE / 'masks';   MASKS.mkdir(exist_ok=True)
print('staging:', STAGE)
print('manifest exists:', (STAGE / 'manifest.csv').exists())
print('tiles  :', len(list(TILES.glob('*.tif'))))
print('labels :', len(list(LABELS.glob('*.shp'))))


In [ ]:
# --- Cell 3: read manifest, mouza-disjoint train/val split ---
import csv, random
rows = []
with open(STAGE / 'manifest.csv', newline='', encoding='utf-8') as fh:
    for r in csv.DictReader(fh):
        rows.append({'x': int(r['tile_x_km']), 'y': int(r['tile_y_km']),
                     'tif': STAGE / r['tile_tif_relpath'],
                     'mouzas': r['mouza_label_stems'].split('|')})
print(f'manifest rows: {len(rows)}')

all_mouzas = sorted({m for r in rows for m in r['mouzas']})
random.Random(CFG['split']['seed']).shuffle(all_mouzas)
n_val = max(1, int(len(all_mouzas) * CFG['split']['val_mouza_fraction']))
if len(all_mouzas) - n_val < 1:
    n_val = len(all_mouzas) - 1
val_mouzas   = set(all_mouzas[:n_val])
train_mouzas = set(all_mouzas[n_val:])
print(f'Train mouzas: {len(train_mouzas)}  Val mouzas: {len(val_mouzas)}')
print('  val =', sorted(val_mouzas))


In [ ]:
# --- Cell 4: rasterise polygons -> 3px boundary masks (cached) ---
import numpy as np, rasterio, cv2, shapefile
from tqdm.auto import tqdm

THICK = CFG['dataset']['boundary_thickness_px']

def read_polys_pyshp(shp_path):
    rdr = shapefile.Reader(str(shp_path))
    polys = []
    for shp in rdr.shapes():
        pts = shp.points
        parts = list(shp.parts) + [len(pts)]
        for i in range(len(parts) - 1):
            polys.append(pts[parts[i]: parts[i+1]])
    return polys

def mask_for_tile(tif_path, mouza_stems, thick=THICK):
    with rasterio.open(tif_path) as ds:
        H, W = ds.height, ds.width
        T = ds.transform
    mask = np.zeros((H, W), dtype=np.uint8)
    for stem in mouza_stems:
        shp_path = LABELS / f'{stem}.shp'
        if not shp_path.exists(): print(f'  WARN missing label: {shp_path.name}'); continue
        for ring in read_polys_pyshp(shp_path):
            xs = np.array([p[0] for p in ring], dtype=np.float64)
            ys = np.array([p[1] for p in ring], dtype=np.float64)
            cols, rows_ = ~T * (xs, ys)
            pts = np.stack([cols, rows_], axis=1).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(mask, [pts], isClosed=True, color=255, thickness=thick)
    return mask

for r in tqdm(rows, desc='rasterising'):
    mp = MASKS / (Path(r['tif']).stem + '_mask.png')
    if mp.exists() and mp.stat().st_size > 0: continue
    m = mask_for_tile(r['tif'], r['mouzas'])
    cv2.imwrite(str(mp), m)
print('masks ->', MASKS)


In [ ]:
# --- Cell 5: build patch lists (mouza-disjoint split) ---
PATCH  = CFG['dataset']['patch_size']
STRIDE = CFG['dataset']['stride']

def is_train_tile(r): return all(m in train_mouzas for m in r['mouzas'])
def is_val_tile(r):   return all(m in val_mouzas   for m in r['mouzas'])

train_patches, val_patches = [], []
for r in rows:
    target = train_patches if is_train_tile(r) else (val_patches if is_val_tile(r) else None)
    if target is None: continue
    with rasterio.open(r['tif']) as ds: H, W = ds.height, ds.width
    mp = MASKS / (Path(r['tif']).stem + '_mask.png')
    for top in range(0, H - PATCH + 1, STRIDE):
        for left in range(0, W - PATCH + 1, STRIDE):
            target.append({'tif': str(r['tif']), 'mask': str(mp), 'top': top, 'left': left})

if CFG['dataset']['drop_empty_patches']:
    print('Filtering empty patches...')
    cache = {}
    def has_signal(p):
        if p['mask'] not in cache: cache[p['mask']] = cv2.imread(p['mask'], cv2.IMREAD_UNCHANGED)
        m = cache[p['mask']]
        return m[p['top']:p['top']+PATCH, p['left']:p['left']+PATCH].any()
    train_patches = [p for p in train_patches if has_signal(p)]
    val_patches   = [p for p in val_patches   if has_signal(p)]

print(f'Train patches: {len(train_patches)}   Val patches: {len(val_patches)}')
assert train_patches, 'No train patches. Lower val_mouza_fraction or stage more mouzas.'
assert val_patches,   'No val patches. Raise val_mouza_fraction or stage more mouzas.'


In [ ]:
# --- Cell 6: copy tiles + masks to Colab's local SSD ---
import shutil, time
LOCAL = Path('/content/local_train')
LOCAL_TILES = LOCAL / 'tiles'; LOCAL_TILES.mkdir(parents=True, exist_ok=True)
LOCAL_MASKS = LOCAL / 'masks'; LOCAL_MASKS.mkdir(parents=True, exist_ok=True)

unique_tifs = sorted({Path(p['tif']) for p in train_patches + val_patches})
t0 = time.time()
for src in unique_tifs:
    dst = LOCAL_TILES / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print(f'  copying {src.name} ({src.stat().st_size/1e6:.0f} MB) ...')
        shutil.copy2(src, dst)
    mask_src = MASKS / (src.stem + '_mask.png')
    mask_dst = LOCAL_MASKS / mask_src.name
    if not mask_dst.exists() or mask_dst.stat().st_size != mask_src.stat().st_size:
        shutil.copy2(mask_src, mask_dst)

for p in train_patches + val_patches:
    p['tif']  = str(LOCAL_TILES / Path(p['tif']).name)
    p['mask'] = str(LOCAL_MASKS / Path(p['mask']).name)

local_gb = sum(f.stat().st_size for f in LOCAL_TILES.glob('*.tif')) / 1e9
print(f'\nLocal cache: {local_gb:.2f} GB in {time.time()-t0:.0f}s')


In [ ]:
# --- Cell 7: pre-extract ALL patches into RAM as numpy arrays ---
# v1.1 speed fix: previously every DataLoader __getitem__ opened a 300 MB TIFF
# and read a 512x512 window -> ~10 min/epoch on L4 (GPU starved, all I/O).
# Now we read each tile ONCE here, slice all its patches from RAM, and the
# DataLoader becomes zero-I/O. Per-epoch time on L4 drops to ~20-30 sec.
#
# Memory cost (rule of thumb): N_patches * (PATCH*PATCH*3 + PATCH*PATCH) bytes.
# 1856 patches @ 512^2 -> ~2 GB. Colab gives 12+ GB RAM; well within budget.

def extract_all(patches, name):
    n = len(patches)
    imgs  = np.empty((n, PATCH, PATCH, 3), dtype=np.uint8)
    masks = np.empty((n, PATCH, PATCH),    dtype=np.uint8)
    by_tif = {}
    for i, p in enumerate(patches):
        by_tif.setdefault(p['tif'], []).append((i, p))
    for tif_path, items in tqdm(by_tif.items(), desc=f'extract {name}'):
        with rasterio.open(tif_path) as ds:
            tile = ds.read([1, 2, 3])         # (3, H, W)
        tile = np.transpose(tile, (1, 2, 0))  # (H, W, 3)
        mask_full = cv2.imread(items[0][1]['mask'], cv2.IMREAD_UNCHANGED)
        for i, p in items:
            imgs[i]  = tile[p['top']:p['top']+PATCH, p['left']:p['left']+PATCH]
            masks[i] = (mask_full[p['top']:p['top']+PATCH, p['left']:p['left']+PATCH] > 0).astype(np.uint8)
    return imgs, masks

print(f'Pre-extracting {len(train_patches)} train + {len(val_patches)} val patches...')
train_imgs, train_masks = extract_all(train_patches, 'train')
val_imgs,   val_masks   = extract_all(val_patches,   'val')
total_gb = (train_imgs.nbytes + train_masks.nbytes + val_imgs.nbytes + val_masks.nbytes) / 1e9
print(f'\nCached in RAM: train {train_imgs.shape}  val {val_imgs.shape}   = {total_gb:.2f} GB')


In [ ]:
# --- Cell 8: in-memory Dataset + DataLoaders ---
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def build_aug(train):
    A_CFG = CFG['augment']
    if train:
        return A.Compose([
            A.HorizontalFlip(p=A_CFG['hflip_p']),
            A.VerticalFlip(p=A_CFG['vflip_p']),
            A.RandomRotate90(p=A_CFG['rot90_p']),
            A.RandomBrightnessContrast(brightness_limit=A_CFG['brightness'], contrast_limit=A_CFG['contrast'], p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class InMemoryDataset(Dataset):
    def __init__(self, imgs, masks, train):
        self.imgs = imgs; self.masks = masks; self.aug = build_aug(train)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        out = self.aug(image=self.imgs[i], mask=self.masks[i].astype(np.float32))
        return out['image'], out['mask'].unsqueeze(0)

train_ds = InMemoryDataset(train_imgs, train_masks, train=True)
val_ds   = InMemoryDataset(val_imgs,   val_masks,   train=False)
BS = CFG['train']['batch_size']
# num_workers=0 is fastest for in-RAM data (no worker-process IPC overhead)
train_dl = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)
print(f'train batches: {len(train_dl)}   val batches: {len(val_dl)}')


In [ ]:
# --- Cell 9: model + train loop ---
import segmentation_models_pytorch as smp
from torch.amp import autocast, GradScaler

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

M = CFG['model']
model = getattr(smp, M['arch'])(
    encoder_name=M['encoder'], encoder_weights=M['encoder_weights'],
    in_channels=M['in_channels'], classes=M['classes'],
).to(DEVICE)

bce  = torch.nn.BCEWithLogitsLoss()
dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
BCE_W, DICE_W = CFG['train']['loss']['bce_weight'], CFG['train']['loss']['dice_weight']
def loss_fn(logits, y): return BCE_W * bce(logits, y) + DICE_W * dice(logits, y)

opt = torch.optim.AdamW(model.parameters(), lr=CFG['train']['lr'], weight_decay=CFG['train']['weight_decay'])
EPOCHS = CFG['train']['epochs']
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = GradScaler('cuda', enabled=CFG['train']['amp'])

def iou_from_logits(logits, y, thr=0.5, eps=1e-7):
    p = (torch.sigmoid(logits) > thr).float()
    inter = (p * y).sum(dim=(1,2,3))
    union = ((p + y) >= 1).float().sum(dim=(1,2,3))
    return ((inter + eps) / (union + eps)).mean().item()

CKPT = STAGE / 'checkpoints'; CKPT.mkdir(exist_ok=True)
best_iou = -1.0

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    model.train(); tr_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=CFG['train']['amp']):
            logits = model(x); loss = loss_fn(logits, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tr_loss += loss.item() * x.size(0)
    tr_loss /= max(1, len(train_ds))

    model.eval(); v_loss, v_iou, n = 0.0, 0.0, 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x); loss = loss_fn(logits, y)
            v_loss += loss.item() * x.size(0); v_iou += iou_from_logits(logits, y) * x.size(0); n += x.size(0)
    v_loss /= max(1, n); v_iou /= max(1, n)
    sched.step()

    dt = time.time() - t_ep
    print(f'ep {epoch:02d}/{EPOCHS}  train_loss={tr_loss:.4f}  val_loss={v_loss:.4f}  val_iou={v_iou:.4f}  lr={opt.param_groups[0]["lr"]:.2e}  {dt:.0f}s')
    if v_iou > best_iou:
        best_iou = v_iou
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_iou': v_iou, 'cfg': CFG}, CKPT / 'best.pt')
        print(f'  saved best (val_iou={v_iou:.4f})')

print(f'\nDone. best val IoU = {best_iou:.4f}')


In [ ]:
# --- Cell 10: qualitative eval on a held-out val tile ---
import matplotlib.pyplot as plt

val_tile_rows = [r for r in rows if all(m in val_mouzas for m in r['mouzas'])]
assert val_tile_rows, 'No val tile found; widen val_mouza_fraction or stage more mouzas.'
vr = val_tile_rows[0]
vr_tif = LOCAL_TILES / Path(vr['tif']).name if (LOCAL_TILES / Path(vr['tif']).name).exists() else vr['tif']
vr_mask = LOCAL_MASKS / (Path(vr['tif']).stem + '_mask.png') if (LOCAL_MASKS / (Path(vr['tif']).stem + '_mask.png')).exists() else (MASKS / (Path(vr['tif']).stem + '_mask.png'))
print('Visualising:', Path(vr_tif).name, '  mouzas:', vr['mouzas'])

with rasterio.open(vr_tif) as ds:
    H, W = ds.height, ds.width
    top, left = H // 2 - 512, W // 2 - 512
    crop = ds.read([1, 2, 3], window=rasterio.windows.Window(left, top, 1024, 1024))
crop = np.transpose(crop, (1, 2, 0))

model.eval()
norm = A.Compose([A.Normalize(), ToTensorV2()])
with torch.no_grad():
    pred = np.zeros((1024, 1024), dtype=np.float32)
    cnt  = np.zeros((1024, 1024), dtype=np.float32)
    for top2 in range(0, 1024 - PATCH + 1, STRIDE):
        for left2 in range(0, 1024 - PATCH + 1, STRIDE):
            sub = crop[top2:top2+PATCH, left2:left2+PATCH]
            x = norm(image=sub)['image'].unsqueeze(0).to(DEVICE)
            p = torch.sigmoid(model(x))[0,0].cpu().numpy()
            pred[top2:top2+PATCH, left2:left2+PATCH] += p
            cnt[top2:top2+PATCH, left2:left2+PATCH]  += 1
    pred = pred / np.maximum(cnt, 1)

gt_full = cv2.imread(str(vr_mask), cv2.IMREAD_UNCHANGED)
gt = gt_full[top:top+1024, left:left+1024]

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(crop); ax[0].set_title('RGB')
ax[1].imshow(crop); ax[1].imshow(gt, cmap='Reds', alpha=0.5); ax[1].set_title('Ground-truth boundaries (red)')
ax[2].imshow(crop); ax[2].imshow(pred, cmap='Blues', alpha=0.6); ax[2].set_title('Predicted boundary probability')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()
